In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split, TunedThresholdClassifierCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, make_scorer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

In [3]:
heart_df = pd.read_csv("../data/brfss_2023_heart_risk_clean_extended.csv") 
heart_df

,ever_heart_attack,ever_coronary_heart_disease,arthritis,high_cholesterol,ever_smoked_100_cigs,exercise_past_30_days,age_group,sex,poor_physical_health_days,poor_mental_health_days,...,stroke_history,difficulty_walking,any_cvd,checkup_recency,current_smoker,bmi,general_health_label,high_blood_pressure_label,diabetes_label,alcohol_days_month
0,0,0.0,0.0,0.0,0.0,0.0,45-49,Female,0.0,0.0,...,0.0,1.0,0,1–2 years,0.0,30.47,Very good,Yes,Yes,0.0
1,0,0.0,1.0,1.0,0.0,1.0,45-49,Female,0.0,0.0,...,0.0,0.0,0,1–2 years,0.0,28.56,Very good,Yes,No,0.0
2,0,0.0,1.0,1.0,1.0,1.0,45-49,Female,6.0,2.0,...,0.0,1.0,0,≤1 year,0.0,22.31,Fair,Yes,No,0.0
3,0,0.0,1.0,0.0,0.0,1.0,45-49,Female,2.0,0.0,...,0.0,1.0,0,2–5 years,0.0,27.44,Very good,No,No,0.0
4,0,0.0,1.0,0.0,0.0,1.0,45-49,Female,0.0,0.0,...,0.0,1.0,0,≤1 year,0.0,25.85,Fair,Yes,Yes,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
430750,0,0.0,0.0,1.0,0.0,1.0,45-49,Male,12.0,30.0,...,NaN,0.0,0,≤1 year,0.0,29.21,Good,Yes,No,22.0
430751,0,0.0,0.0,0.0,0.0,0.0,25-29,Female,0.0,0.0,...,0.0,0.0,0,≤1 year,0.0,24.96,Very good,No,No,0.0
430752,0,0.0,0.0,1.0,0.0,1.0,35-39,Female,10.0,0.0,...,0.0,0.0,0,≤1 year,0.0,34.38,Very good,No,No,NaN
430753,0,0.0,0.0,1.0,0.0,1.0,45-49,Female,0.0,0.0,...,0.0,0.0,0,≤1 year,0.0,23.86,Good,Yes,Yes,0.0


In [4]:
# from ydata_profiling import ProfileReport
# ProfileReport(heart_df.iloc[1:10000,:])

In [5]:
heart_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 430755 entries, 0 to 430754
Data columns (total 25 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   ever_heart_attack             430755 non-null  int64  
 1   ever_coronary_heart_disease   427159 non-null  float64
 2   arthritis                     428503 non-null  float64
 3   high_cholesterol              376398 non-null  float64
 4   ever_smoked_100_cigs          408539 non-null  float64
 5   exercise_past_30_days         429589 non-null  float64
 6   age_group                     430755 non-null  object 
 7   sex                           430755 non-null  object 
 8   poor_physical_health_days     420255 non-null  float64
 9   poor_mental_health_days       422854 non-null  float64
 10  activity_limited_health_days  420255 non-null  float64
 11  current_asthma                429333 non-null  float64
 12  skin_cancer_history           428148 non-nul

In [6]:
binary_condition_map = {
    "No": 0,
    "No, but Borderline": 0,
    "Yes, during pregnancy": 0,
    "Yes": 1
}
heart_df["diabetes_label"] = (
    heart_df["diabetes_label"]
    .map(binary_condition_map)
    .astype("Int64")
)

heart_df["high_blood_pressure_label"] = (
    heart_df["high_blood_pressure_label"]
    .map(binary_condition_map)
    .astype("Int64")
)

# Train-Test split

In [7]:
from sklearn.model_selection import train_test_split

X = heart_df.drop(['ever_heart_attack', 'ever_coronary_heart_disease', 'any_cvd'], axis=1)
y = heart_df['any_cvd']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Preprocessor with one-hot encoding

In [8]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer

bin_cols = [
    "ever_smoked_100_cigs",
    "exercise_past_30_days",
    "current_asthma",
    "skin_cancer_history",
    "copd_history",
    "kidney_disease_history",
    "stroke_history",
    "difficulty_walking",
    "high_cholesterol",
    "current_smoker",
    "diabetes_label", 
    "high_blood_pressure_label",
    "arthritis"
]


num_cols = [
    "bmi",
    "alcohol_days_month",
    "poor_physical_health_days",
    "poor_mental_health_days",
    "activity_limited_health_days"
]

cat_cols = [
    "sex",
    "age_group",
    "general_health_label",
    "checkup_recency",
    #"high_blood_pressure_label",
    #"diabetes_label",
]

# Numeric pipeline
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

bin_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

# Categorical pipeline
cat_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Combine into preprocessor
preprocessor = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("bin", bin_pipe, bin_cols),
    ("cat", cat_pipe, cat_cols)
])

preprocessor

,transformers,"[('num', ...), ('bin', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


# Preprocessor with both one-hot & ordinal encoding

In [9]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer


# -----------------------------
# 1. ORDINAL COLUMNS & CATEGORIES
# -----------------------------

ordinal_cols = ["age_group",
                "general_health_label", 
                "checkup_recency",
                # "diabetes_label", 
                # "high_blood_pressure_label"
                ]

ordinal_categories = [

    # AgeCategory order
    [ "80 or older", "75-79", "70-74", "65-69", "60-64", "55-59", "50-54",
    "45-49", "40-44", "35-39", "30-34", "25-29", "18-24"],

    # GenHealth order
    ["Poor", "Fair", "Good", "Very good", "Excellent"],

    # checkup
    ["≥5 years" ,"2–5 years" ,"1–2 years" ,"≤1 year"] 


    # # Diabetic order
    # ["No", "No, but Borderline", "Yes, during pregnancy", "Yes"],

    # # cholesterol order
    # ["No", "No, but Borderline", "Yes, during pregnancy", "Yes"]
]


# -----------------------------
# 2. Nominal categorical columns
# -----------------------------

nominal_cols = [ 'sex']
bin_cols = [
    "ever_smoked_100_cigs",
    "exercise_past_30_days",
    "current_asthma",
    "skin_cancer_history",
    "copd_history",
    "kidney_disease_history",
    "stroke_history",
    "difficulty_walking",
    "high_cholesterol",
    "current_smoker",
    "diabetes_label", 
    "high_blood_pressure_label",
    "arthritis"
]


# -----------------------------
# 3. Numeric columns
# -----------------------------
num_cols = [
    "bmi",
    "alcohol_days_month",
    "poor_physical_health_days",
    "poor_mental_health_days",
    "activity_limited_health_days"
]

# -----------------------------
# 4. PIPELINES
# -----------------------------

# Numeric pipeline ✅
num_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Binary pipeline ✅
bin_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

# Nominal categorical pipeline ✅
nominal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# Ordinal categorical pipeline ✅
ordinal_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("ord", OrdinalEncoder(categories=ordinal_categories))
])

# -----------------------------
# 5. COMBINED PREPROCESSOR
# -----------------------------

preprocessor_ord = ColumnTransformer([
    ("num", num_pipe, num_cols),
    ("bin", bin_pipe, bin_cols),
    ("ord", ordinal_pipe, ordinal_cols),
    ("nom", nominal_pipe, nominal_cols)
])

preprocessor_ord

,transformers,"[('num', ...), ('bin', ...), ...]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,missing_values,nan
,strategy,'median'
,fill_value,None


# Function to test with different models

In [10]:
def recall_accuracy_scorer(y_true, y_pred):
    recall = recall_score(y_true, y_pred, pos_label=1)
    accuracy = accuracy_score(y_true,y_pred)
    combined_score = 0.8 * recall + 0.2 * accuracy  # Adjust weights as needed
    return combined_score

combined_scorer = make_scorer(recall_accuracy_scorer)

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, recall_score, confusion_matrix, balanced_accuracy_score, roc_auc_score
from sklearn.feature_selection import SelectFromModel

models_score = []

def evaluate_model(model_name,  probs, y_test, threshold=0.5):
    preds = (probs >= threshold).astype(int)      # convert to 0/1 based on threshold
    acc = accuracy_score(y_test, preds)
    acc_bal = balanced_accuracy_score(y_test, preds, adjusted=True)     
    prec = precision_score(y_test, preds)
    rec = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    roc_auc = roc_auc_score(y_test, preds)
    
    print("\n============================")
    print(model_name)
    print("============================")

    
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    print("Confusion Matrix:")
    print(f"TP={tp}, FN={fn}, FP={fp}, TN={tn}")
    print("Recall =", recall_score(y_test, preds))

    models_score.append({
            "Model": model_name,
            "Threshold":threshold,
            "TN":tn,
            "FP": fp,
            "FN": fn,
            "TP": tp,
            "Accuracy": acc,
            "BalancedAccuracy":acc_bal,
            "Precision": prec,
            "Recall": rec,
            "F1 Score": f1,
            "ROC-AUC":  roc_auc
        })
    results_df = pd.DataFrame(models_score).sort_values(by="ROC-AUC", ascending=False).reset_index(drop=True)
    return results_df

# Logistic Regression

## Initial fit

In [12]:
from sklearn.linear_model import LogisticRegression
selector = SelectFromModel(
    RandomForestClassifier(n_estimators=200, random_state=42),
    threshold="median"      # keep features above median importance
)
model = LogisticRegression(
    C=10,                    # less regularization
    penalty='l2',
    solver='saga',
    class_weight='balanced',
    max_iter=1000,           # may need fewer iterations
    random_state=42
)

pipe = Pipeline([
    ("prep", preprocessor_ord),
    ("select", selector),
    ("model", model)
])

pipe.fit(X_train, y_train)
probs = pipe.predict_proba(X_test)[:, 1]      # probability of class 1

evaluate_model(model.__class__.__name__, probs, y_test)


LogisticRegression
Confusion Matrix:
TP=5725, FN=1460, FP=24222, TN=54744
Recall = 0.7967988865692415


,Model,Threshold,TN,FP,FN,TP,Accuracy,BalancedAccuracy,Precision,Recall,F1 Score,ROC-AUC
0,LogisticRegression,0.5,54744,24222,1460,5725,0.701896,0.490059,0.191171,0.796799,0.308359,0.74503


## Randomized search

In [ ]:
# from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
# from scipy.stats import randint, uniform

# pipe = Pipeline([
#     ("prep", preprocessor_ord),
#     # ("select", selector),
#     # # ("model", model)
# # ])

# # # Parameter distributions
# # param_dist = {
# #     # Preprocessing
# #     'prep__nom__onehot__drop': [None, 'first'],
# #     'prep__nom__onehot__sparse_output': [True, False],
    
# #     # Feature selection
# #     'select__estimator__n_estimators': randint(100, 500),
# #     'select__threshold': ['0.1*mean', 'mean', 'median'],

# #     # Logistic Regression
# #     'model__C': uniform(0.1, 50),
# #     'model__solver': ['saga'],  # saga works with l1/l2 + large data
# #     'model__penalty': ['l2', 'l1'],
# #     'model__class_weight': ['balanced', None]
# # }

# # # Stratified CV for imbalanced data
# # cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# # Randomized search
# rand_search = RandomizedSearchCV(
#     pipe,
#     param_distributions=param_dist,
#     n_iter=10,          # number of random combinations
#     scoring='roc_auc',  # good for imbalanced data
#     cv=cv,
#     verbose=2,
#     random_state=42,
#     n_jobs=-1
# )

# # Fit
# rand_search.fit(X_train, y_train)

# # Best results
# print(f"Best params: {rand_search.best_params_}")
# print(f"Best ROC-AUC: {rand_search.best_score_:.4f}")

# # Predict probabilities
# probs = rand_search.predict_proba(X_test)[:, 1]
# evaluate_model(rand_search.best_estimator_.named_steps['model'].__class__.__name__, probs, y_test)


## Tune threshold

In [ ]:
# from sklearn.linear_model import LogisticRegression
# selector = SelectFromModel(
#     RandomForestClassifier(n_estimators=288, random_state=42),
#     threshold="0.1*mean"      # keep features above median importance
# )

# model = LogisticRegression(
#     C=18.8,                    # less regularization
#     penalty='l2',
#     solver='saga',
#     class_weight='balanced',
#     max_iter=1000,           # may need fewer iterations
#     random_state=42
# )

# # Use same pipeline structure
# pipe = Pipeline([
#     ("prep", preprocessor_ord),
#     ("select", selector),
#     ("model", model)
# ])

# pipe.fit(X_train, y_train)
# probs = pipe.predict_proba(X_test)[:, 1]
# evaluate_model(model.__class__.__name__, probs, y_test)

In [ ]:
# tuned_model = TunedThresholdClassifierCV(
#     estimator=pipe,#rfc_search.best_estimator_,
#     scoring=combined_scorer,
#     store_cv_results=True  # necessary to inspect all results
# )

# _ = tuned_model.fit(X_train, y_train)
# tuned_model.best_threshold_ 
# evaluate_model([model.__class__.__name__ + " " + str(round(tuned_model.best_threshold_,4) )],
#                 probs, y_test, tuned_model.best_threshold_ )

# Decision Tree

## First run

In [13]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(
    max_depth=10,                    # Prevent overfitting on 320K samples
    min_samples_split=1000,          # ~0.3% of data (320K/320)
    min_samples_leaf=500,            # ~0.15% of data
    class_weight='balanced',         # Handle 8% imbalance
    random_state=42,
    criterion='gini',                # Start with gini (faster)
    max_features='sqrt',             # Adds randomness, reduces overfitting
    min_impurity_decrease=0.0001     # Prune small improvements
)

# Use same pipeline structure
pipe = Pipeline([
    ("prep", preprocessor),
    ("model", model)  # Remove feature selector - trees handle this
])

pipe.fit(X_train, y_train)
probs = pipe.predict_proba(X_test)[:, 1]
evaluate_model(model.__class__.__name__, probs, y_test)


DecisionTreeClassifier
Confusion Matrix:
TP=5724, FN=1461, FP=25163, TN=53803
Recall = 0.7966597077244258


,Model,Threshold,TN,FP,FN,TP,Accuracy,BalancedAccuracy,Precision,Recall,F1 Score,ROC-AUC
0,LogisticRegression,0.5,54744,24222,1460,5725,0.701896,0.490059,0.191171,0.796799,0.308359,0.745030
1,DecisionTreeClassifier,0.5,53803,25163,1461,5724,0.690961,0.478004,0.185321,0.796660,0.300693,0.739002


## Randomized Search

In [ ]:
# from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
# from scipy.stats import randint, uniform

# # Define distributions for random sampling
# param_distributions = {
#     'model__max_depth': randint(8, 25),                    # Sample from 8-24
#     'model__min_samples_split': randint(500, 3000),        # Sample from 500-2999
#     'model__min_samples_leaf': randint(250, 1500),         # Sample from 250-1499
#     'model__criterion': ['gini', 'entropy'],
#     'model__max_features': ['sqrt', 'log2', None],
#     'model__class_weight': ['balanced', {0: 1, 1: 12}],
#     'model__min_impurity_decrease': uniform(0, 0.002)      # Sample from 0-0.002
# }

# cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)  # Reduced to 3 folds

# random_search = RandomizedSearchCV(
#     pipe,
#     param_distributions,
#     n_iter=30,              # Try 30 random combinations (vs 1000+ in full grid)
#     cv=cv,
#     scoring='roc_auc',
#     n_jobs=-1,
#     verbose=2,
#     random_state=42
# )

# random_search.fit(X_train, y_train)
# print(f"Best params: {random_search.best_params_}")
# print(f"Best ROC-AUC: {random_search.best_score_:.4f}")
# probs = random_search.predict_proba(X_test)[:, 1]
# evaluate_model(model.__class__.__name__, probs, y_test)

## Grid search around best regions

In [ ]:
# param_grid = {
#     'model__max_depth': [10, 12],
#     'model__min_samples_split': [2500],
#     'model__min_samples_leaf': [350, 400],
#     'model__criterion': ['gini'],
#     'model__max_features': [ None],
#     'model__class_weight': ['balanced'],
#     'model__min_impurity_decrease': [ 0.000001, 0.00001]
# }

# from sklearn.model_selection import StratifiedKFold, GridSearchCV

# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# grid = GridSearchCV(
#     pipe,
#     param_grid,
#     cv=cv,
#     scoring='roc_auc',
#     n_jobs=-1,
#     verbose=2
# )

# grid.fit(X_train, y_train)
# probs = grid.predict_proba(X_test)[:, 1]
# evaluate_model(model.__class__.__name__, probs, y_test)

## Tune Threshold

In [ ]:
# model = DecisionTreeClassifier(
#     max_depth=10,                    # Prevent overfitting on 320K samples
#     min_samples_split=2500,          # ~0.3% of data (320K/320)
#     min_samples_leaf=400,            # ~0.15% of data
#     class_weight='balanced',         # Handle 8% imbalance
#     random_state=42,
#     criterion='gini',                # Start with gini (faster)
#     max_features=None,             # Adds randomness, reduces overfitting
#     min_impurity_decrease=0.00001     # Prune small improvements
# )

# # Use same pipeline structure
# dt_pipe = Pipeline([
#     ("prep", preprocessor),
#     ("model", model)  # Remove feature selector - trees handle this
# ])

# pipe.fit(X_train, y_train)
# probs = pipe.predict_proba(X_test)[:, 1]
# evaluate_model(model.__class__.__name__, probs, y_test)

In [ ]:
# dt_tuned_model = TunedThresholdClassifierCV(
#     estimator=dt_pipe,#rfc_search.best_estimator_,
#     scoring=combined_scorer,
#     store_cv_results=True  # necessary to inspect all results
# )

# _ = dt_tuned_model.fit(X_train, y_train)
# dt_tuned_model.best_threshold_ 
# evaluate_model([model.__class__.__name__ + " " + str(round(dt_tuned_model.best_threshold_,4) )],
#                 probs, y_test, dt_tuned_model.best_threshold_ )

In [ ]:
# # calculate roc curves
# from sklearn.metrics import roc_curve
# import matplotlib.pyplot as plt
# %matplotlib inline

# probs = pipe.predict_proba(X_test)[:, 1]  # probability for class 1
# fpr, tpr, thresholds = roc_curve(y_test, probs)
# # plot the roc curve for the model
# plt.plot(fpr, tpr, marker='.', label=model.__class__.__name__)
# # axis labels
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# # show the legend
# plt.legend()
# # show the plot
# plt.show()

In [14]:
# 1) Get feature names after preprocessing
feature_names = pipe.named_steps["prep"].get_feature_names_out()

# 2) Get importances from the decision tree
importances = pipe.named_steps["model"].feature_importances_

# 3) Put into a DataFrame and sort
fi = pd.DataFrame(
    {"feature": feature_names, "importance": importances}
).sort_values("importance", ascending=False)

print(fi.head(20))   # top 20 features

                                feature  importance
13                bin__high_cholesterol    0.275195
16       bin__high_blood_pressure_label    0.210499
12              bin__difficulty_walking    0.116205
25                 cat__age_group_45-49    0.102713
18                      cat__sex_Female    0.040702
2        num__poor_physical_health_days    0.040268
11                  bin__stroke_history    0.034189
9                     bin__copd_history    0.030667
27       cat__general_health_label_Fair    0.021369
15                  bin__diabetes_label    0.020301
17                       bin__arthritis    0.014857
20                 cat__age_group_18-24    0.014292
24                 cat__age_group_40-44    0.013268
29       cat__general_health_label_Poor    0.012298
30  cat__general_health_label_Very good    0.011908
33         cat__checkup_recency_≤1 year    0.009414
22                 cat__age_group_30-34    0.007850
23                 cat__age_group_35-39    0.004392
26  cat__gen

# Random Forest

In [15]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42
)
rf_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", rf)  # No feature selector - boosting handles this
])
rf_pipe.fit(X_train, y_train)
# Evaluate
probs = rf_pipe.predict_proba(X_test)[:, 1]
evaluate_model(model.__class__.__name__, probs, y_test)


DecisionTreeClassifier
Confusion Matrix:
TP=5790, FN=1395, FP=21489, TN=57477
Recall = 0.8058455114822547


,Model,Threshold,TN,FP,FN,TP,Accuracy,BalancedAccuracy,Precision,Recall,F1 Score,ROC-AUC
0,DecisionTreeClassifier,0.5,57477,21489,1395,5790,0.734373,0.533716,0.212251,0.805846,0.336003,0.766858
1,LogisticRegression,0.5,54744,24222,1460,5725,0.701896,0.490059,0.191171,0.796799,0.308359,0.745030
2,DecisionTreeClassifier,0.5,53803,25163,1461,5724,0.690961,0.478004,0.185321,0.796660,0.300693,0.739002


In [16]:
threshold=0.19
evaluate_model(model.__class__.__name__ , probs, y_test, threshold)


DecisionTreeClassifier
Confusion Matrix:
TP=6975, FN=210, FP=48912, TN=30054
Recall = 0.9707724425887265


,Model,Threshold,TN,FP,FN,TP,Accuracy,BalancedAccuracy,Precision,Recall,F1 Score,ROC-AUC
0,DecisionTreeClassifier,0.50,57477,21489,1395,5790,0.734373,0.533716,0.212251,0.805846,0.336003,0.766858
1,LogisticRegression,0.50,54744,24222,1460,5725,0.701896,0.490059,0.191171,0.796799,0.308359,0.745030
2,DecisionTreeClassifier,0.50,53803,25163,1461,5724,0.690961,0.478004,0.185321,0.796660,0.300693,0.739002
3,DecisionTreeClassifier,0.19,30054,48912,210,6975,0.429815,0.351367,0.124805,0.970772,0.221176,0.675683


# Gradient boost

## Initial Fit

In [17]:
model = GradientBoostingClassifier(
    n_estimators=200,                # Moderate number of trees
    learning_rate=0.1,               # Standard learning rate
    max_depth=5,                     # Shallow trees (typical for boosting)
    min_samples_split=1000,
    min_samples_leaf=500,
    subsample=0.8,                   # Use 80% of data per tree (faster + regularization)
    max_features='sqrt',
    random_state=42,
)
pipe = Pipeline([
    ("prep", preprocessor),
    ("model", model)  # No feature selector - boosting handles this
])
pipe.fit(X_train, y_train)
# Evaluate
probs = pipe.predict_proba(X_test)[:, 1]
evaluate_model(model.__class__.__name__, probs, y_test)


GradientBoostingClassifier
Confusion Matrix:
TP=569, FN=6616, FP=436, TN=78530
Recall = 0.07919276270006959


,Model,Threshold,TN,FP,FN,TP,Accuracy,BalancedAccuracy,Precision,Recall,F1 Score,ROC-AUC
0,DecisionTreeClassifier,0.50,57477,21489,1395,5790,0.734373,0.533716,0.212251,0.805846,0.336003,0.766858
1,LogisticRegression,0.50,54744,24222,1460,5725,0.701896,0.490059,0.191171,0.796799,0.308359,0.745030
2,DecisionTreeClassifier,0.50,53803,25163,1461,5724,0.690961,0.478004,0.185321,0.796660,0.300693,0.739002
3,DecisionTreeClassifier,0.19,30054,48912,210,6975,0.429815,0.351367,0.124805,0.970772,0.221176,0.675683
4,GradientBoostingClassifier,0.50,78530,436,6616,569,0.918144,0.073671,0.566169,0.079193,0.138950,0.536836


## Randomized search

In [ ]:
# from sklearn.ensemble import GradientBoostingClassifier
# from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
# from scipy.stats import randint, uniform

# # Gradient Boosting model
# model = GradientBoostingClassifier(
#     random_state=42,
#     verbose=0
# )

# pipe = Pipeline([
#     ("prep", preprocessor),
#     ("model", model)  # No feature selector - boosting handles this
# ])

# # Parameter distributions for random search
# param_distributions = {
#     'model__n_estimators': randint(100, 500),              # Number of trees
#     'model__learning_rate': uniform(0.01, 0.29),           # 0.01 to 0.3
#     'model__max_depth': randint(3, 10),                    # Shallow trees for boosting
#     'model__min_samples_split': randint(500, 3000),
#     'model__min_samples_leaf': randint(250, 1500),
#     'model__subsample': uniform(0.6, 0.4),                 # 0.6 to 1.0
#     'model__max_features': ['sqrt', 'log2', None],
# }

# cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# random_search = RandomizedSearchCV(
#     pipe,
#     param_distributions,
#     n_iter=20,
#     cv=cv,
#     scoring='roc_auc',
#     n_jobs=-1,
#     verbose=2,
#     random_state=42
# )

# random_search.fit(X_train, y_train)
# print(f"Best params: {random_search.best_params_}")
# print(f"Best ROC-AUC: {random_search.best_score_:.4f}")

# # Evaluate
# probs = random_search.predict_proba(X_test)[:, 1]
# evaluate_model([model.__class__.__name__ + " search"], probs, y_test)

## Tune threshold

In [ ]:
model = GradientBoostingClassifier(
    n_estimators=135,                # Moderate number of trees
    learning_rate=0.05040612177770395,# Standard learning rate
    max_depth=9,                     # Shallow trees (typical for boosting)
    min_samples_split=1040,
    min_samples_leaf=1444,
    subsample=0.9771414282231924,     # Use 80% of data per tree (faster + regularization)
    max_features='sqrt',
    random_state=42,
    verbose=0                        # Show progress
)
pipe = Pipeline([
    ("prep", preprocessor),
    ("model", model)  # No feature selector - boosting handles this
])
pipe.fit(X_train, y_train)
# Evaluate
probs = pipe.predict_proba(X_test)[:, 1]
evaluate_model(model.__class__.__name__, probs, y_test)

tuned_model = TunedThresholdClassifierCV(
    estimator=pipe,#rfc_search.best_estimator_,
    scoring=combined_scorer,
    store_cv_results=True  # necessary to inspect all results
)

_ = tuned_model.fit(X_train, y_train)
tuned_model.best_threshold_ #0.025348676198706305

evaluate_model(model.__class__.__name__, probs, y_test, tuned_model.best_threshold_ )


GradientBoostingClassifier
Confusion Matrix:
TP=489, FN=6696, FP=351, TN=78615
Recall = 0.06805845511482254


In [ ]:
# # calculate roc curves
# from sklearn.metrics import roc_curve
# import matplotlib.pyplot as plt
# %matplotlib inline

# probs = pipe.predict_proba(X_test)[:, 1]  # probability for class 1
# fpr, tpr, thresholds = roc_curve(y_test, probs)
# # plot the roc curve for the model
# plt.plot(fpr, tpr, marker='.', label=model.__class__.__name__)
# # axis labels
# plt.xlabel('False Positive Rate')
# plt.ylabel('True Positive Rate')
# # show the legend
# plt.legend()
# # show the plot
# plt.show()

# XGBoost

In [18]:
from xgboost import XGBClassifier

model =  XGBClassifier(
    eval_metric="logloss",
    max_depth=4,
    n_estimators=200,
    learning_rate=0.1    
)
pipe = Pipeline([
    ("prep", preprocessor_ord),
    ("model", model)  # No feature selector - boosting handles this
])
pipe.fit(X_train, y_train)
# Evaluate
probs = pipe.predict_proba(X_test)[:, 1]
results_df = evaluate_model(model.__class__.__name__, probs, y_test)



XGBClassifier
Confusion Matrix:
TP=551, FN=6634, FP=422, TN=78544
Recall = 0.07668754349338901


## Randomized Search

In [ ]:
# # required imports
# import numpy as np
# from scipy.stats import randint, uniform, loguniform
# from sklearn.pipeline import Pipeline
# from sklearn.model_selection import RandomizedSearchCV
# from xgboost import XGBClassifier

# # Build final pipeline (assumes preprocessor_ord already exists)
# model = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
# pipe = Pipeline([
#     ("preprocessor", preprocessor_ord),   # your ColumnTransformer
#     ("clf", model)
# ])

# # Small but useful param distributions for XGBoost + some preprocessor choices
# param_distributions = {
#     # --- XGBoost model hyperparameters ---
#     "clf__n_estimators": randint(100, 800),            # number of trees
#     "clf__max_depth": randint(3, 8),                  # tree depth
#     "clf__learning_rate": loguniform(0.01, 0.2),      # eta
#     "clf__subsample": uniform(0.6, 0.4),              # [0.6,1.0)
#     "clf__colsample_bytree": uniform(0.4, 0.6),       # [0.4,1.0)
#     "clf__gamma": uniform(0.0, 3),                  # minimum loss reduction
#     "clf__min_child_weight": randint(1, 6),          # regularization via min child weight
#     "clf__reg_alpha": loguniform(1e-8, 10),           # L1 regularization
#     "clf__reg_lambda": loguniform(1e-8, 10),          # L2 regularization

#     # Optional: handle class imbalance (try leaving at 1 or set to ratio)
#     "clf__scale_pos_weight": [1, (len(y_train)-sum(y_train))/max(1,sum(y_train))], 

#     # --- Preprocessor switches (small set of useful alternatives) ---
#     # whether to apply standard scaling vs omit
#     "preprocessor__num__scaler": [StandardScaler(), "passthrough"],
#     # numeric imputer: median (default) vs mean
#     "preprocessor__num__imputer__strategy": ["median", "mean"],

#     # categorical/ordinal imputer: most_frequent vs constant (rare)
#     "preprocessor__nom__imputer__strategy": ["most_frequent", "constant"],
#     "preprocessor__bin__imputer__strategy": ["most_frequent", "constant"],
# }

# # Create the RandomizedSearchCV
# rs = RandomizedSearchCV(
#     estimator=pipe,
#     param_distributions=param_distributions,
#     n_iter=40,                # small but reasonably exploratory
#     scoring="roc_auc",        # use AUC for imbalanced binary classification
#     cv=5,
#     n_jobs=-1,
#     random_state=42,
#     #refit=True
# )

# # Fit (example)
# rs.fit(X_train, y_train)  # consider passing early stopping via fit_params if using xgboost's native
# print(f"Best params: {rs.best_params_}")
# print(f"Best ROC-AUC: {rs.best_score_:.4f}")

# # Evaluate
# model = XGBClassifier(use_label_encoder=False, eval_metric="logloss", random_state=42)
# probs = rs.predict_proba(X_test)[:, 1]
# evaluate_model([model.__class__.__name__ + " search"], probs, y_test)

# Save results table

In [19]:
results_df

,Model,Threshold,TN,FP,FN,TP,Accuracy,BalancedAccuracy,Precision,Recall,F1 Score,ROC-AUC
0,DecisionTreeClassifier,0.50,57477,21489,1395,5790,0.734373,0.533716,0.212251,0.805846,0.336003,0.766858
1,LogisticRegression,0.50,54744,24222,1460,5725,0.701896,0.490059,0.191171,0.796799,0.308359,0.745030
2,DecisionTreeClassifier,0.50,53803,25163,1461,5724,0.690961,0.478004,0.185321,0.796660,0.300693,0.739002
3,DecisionTreeClassifier,0.19,30054,48912,210,6975,0.429815,0.351367,0.124805,0.970772,0.221176,0.675683
4,GradientBoostingClassifier,0.50,78530,436,6616,569,0.918144,0.073671,0.566169,0.079193,0.138950,0.536836
5,XGBClassifier,0.50,78544,422,6634,551,0.918097,0.071343,0.566290,0.076688,0.135082,0.535672


In [20]:
results_df.to_csv('model_comparison_composite_target_brfss2023.csv', index=False)